# 03 — Build the OGG Flight + Weather + Maui Storm Modeling Table

This notebook constructs the first model-ready dataset for Flight Disruption AI.

**Unit of analysis:** one scheduled flight arriving at or departing from OGG.

For each flight we will:
1. create the scheduled OGG-local event timestamp;
2. attach the nearest prior OGG weather observation (within 3 hours);
3. derive clean numeric weather features;
4. tag whether a Maui/OGG-relevant NOAA Storm Event is active at the flight time;
5. save the merged table to `data/processed/`.


In [ ]:
from pathlib import Path
import math
import re
import pandas as pd
import numpy as np

cwd = Path.cwd().resolve()
if (cwd / 'data').exists():
    ROOT = cwd
elif (cwd.parent / 'data').exists():
    ROOT = cwd.parent
else:
    raise FileNotFoundError(f'Cannot locate project root from {cwd}')

BTS_FILE = ROOT / 'data/raw/bts/ogg_flights_2020_2026.csv.gz'
WEATHER_FILE = ROOT / 'data/raw/weather/ogg_lcd_2020_2026.csv.gz'
STORM_FILE = ROOT / 'data/raw/incidents/hawaii_storm_events_2020_2026.csv.gz'
OUT_FILE = ROOT / 'data/processed/ogg_flight_weather_storm_2020_2026.csv.gz'

for p in [BTS_FILE, WEATHER_FILE, STORM_FILE]:
    print(p, 'exists=', p.exists())


## 1. Load source tables


In [ ]:
flights = pd.read_csv(BTS_FILE, low_memory=False)
weather = pd.read_csv(WEATHER_FILE, low_memory=False)
storms = pd.read_csv(STORM_FILE, low_memory=False)

print('Flights:', flights.shape)
print('Weather:', weather.shape)
print('Storm events:', storms.shape)


## 2. Construct the scheduled OGG event timestamp

`FlightDate` is the BTS operating date. For OGG departures we use `CRSDepTime`; for OGG arrivals we use `CRSArrTime`.

This is a first-pass local-time representation. A later refinement can explicitly reconstruct cross-midnight arrival dates where needed.


In [ ]:
def hhmm_to_timedelta(x):
    if pd.isna(x):
        return pd.NaT
    try:
        x = int(float(x))
    except (TypeError, ValueError):
        return pd.NaT
    hour, minute = divmod(x, 100)
    if hour == 24:
        hour = 0
    if not (0 <= hour <= 23 and 0 <= minute <= 59):
        return pd.NaT
    return pd.Timedelta(hours=hour, minutes=minute)

flights['FlightDate'] = pd.to_datetime(flights['FlightDate'], errors='coerce')
flights['direction'] = np.where(flights['Origin'].eq('OGG'), 'departure', 'arrival')
time_col = np.where(flights['direction'].eq('departure'), flights['CRSDepTime'], flights['CRSArrTime'])
flight_td = pd.Series(time_col, index=flights.index).apply(hhmm_to_timedelta)
flights['ogg_sched_dt'] = flights['FlightDate'] + flight_td

print('Missing scheduled OGG timestamps:', flights['ogg_sched_dt'].isna().sum())
display(flights[['FlightDate','Origin','Dest','CRSDepTime','CRSArrTime','direction','ogg_sched_dt']].head())


## 3. Prepare NOAA OGG weather timestamps


In [ ]:
if 'DATE' not in weather.columns:
    raise KeyError('Expected NOAA LCD column DATE was not found.')

weather['weather_dt'] = pd.to_datetime(weather['DATE'], errors='coerce')

if getattr(weather['weather_dt'].dt, 'tz', None) is not None:
    weather['weather_dt'] = weather['weather_dt'].dt.tz_localize(None)

weather = weather.dropna(subset=['weather_dt']).sort_values('weather_dt').reset_index(drop=True)

print('Weather range:', weather['weather_dt'].min(), 'to', weather['weather_dt'].max())
print('Parsed weather rows:', len(weather))


## 4. Create clean numeric weather features

NOAA LCD fields can contain trace markers or quality flags. We keep the original columns and create `_num` versions for the main candidate weather predictors.


In [ ]:
weather_candidates = [
    'HourlyDryBulbTemperature',
    'HourlyDewPointTemperature',
    'HourlyRelativeHumidity',
    'HourlyStationPressure',
    'HourlySeaLevelPressure',
    'HourlyVisibility',
    'HourlyWindDirection',
    'HourlyWindSpeed',
    'HourlyWindGustSpeed',
    'HourlyPrecipitation',
]

def parse_noaa_numeric(v):
    if pd.isna(v):
        return np.nan
    s = str(v).strip()
    if s in {'', 'T', 'M', 'NA', 'nan'}:
        return 0.0 if s == 'T' else np.nan
    m = re.search(r'[-+]?\d*\.?\d+', s)
    return float(m.group()) if m else np.nan

available_weather = [c for c in weather_candidates if c in weather.columns]
for c in available_weather:
    weather[c + '_num'] = weather[c].map(parse_noaa_numeric)

print('Available weather predictors:')
display(pd.DataFrame({'column': available_weather}))


## 5. Match each flight to the nearest prior OGG weather observation

We use backward `merge_asof` with a 3-hour tolerance so that the model never uses a future weather observation.


In [ ]:
flight_merge = flights.dropna(subset=['ogg_sched_dt']).sort_values('ogg_sched_dt').copy()

weather_keep = ['weather_dt'] + available_weather + [c + '_num' for c in available_weather]
weather_merge = weather[weather_keep].copy().sort_values('weather_dt')

merged = pd.merge_asof(
    flight_merge,
    weather_merge,
    left_on='ogg_sched_dt',
    right_on='weather_dt',
    direction='backward',
    tolerance=pd.Timedelta('3h'),
)

merged['weather_age_min'] = (
    merged['ogg_sched_dt'] - merged['weather_dt']
).dt.total_seconds() / 60

print('Merged flight rows:', merged.shape)
print('Flights with matched weather:', merged['weather_dt'].notna().sum())
print('Weather match rate (%):', round(100 * merged['weather_dt'].notna().mean(), 2))
display(merged[['ogg_sched_dt','weather_dt','weather_age_min']].head())


## 6. Identify Maui/OGG-relevant Storm Events

The NOAA Storm Events download is statewide. For this first spatial filter we flag events as Maui-relevant when their text fields reference Maui-area geography, or when an event coordinate lies within roughly 100 km of OGG.

This is intentionally broader than the final production rule and can be refined after inspecting the retained events.


In [ ]:
OGG_LAT, OGG_LON = 20.8986, -156.4305

def haversine_km(lat1, lon1, lat2, lon2):
    try:
        lat1, lon1, lat2, lon2 = map(float, [lat1, lon1, lat2, lon2])
    except (TypeError, ValueError):
        return np.nan
    R = 6371.0088
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dp = math.radians(lat2-lat1)
    dl = math.radians(lon2-lon1)
    a = math.sin(dp/2)**2 + math.cos(p1)*math.cos(p2)*math.sin(dl/2)**2
    return 2*R*math.asin(math.sqrt(a))

text_cols = [c for c in [
    'CZ_NAME','BEGIN_LOCATION','END_LOCATION','EPISODE_NARRATIVE','EVENT_NARRATIVE'
] if c in storms.columns]

if text_cols:
    combined_text = storms[text_cols].fillna('').astype(str).agg(' '.join, axis=1).str.upper()
else:
    combined_text = pd.Series('', index=storms.index)

maui_terms = r'MAUI|KAHULUI|WAILUKU|KIHEI|LAHAINA|HALEAKALA|MAKAWAO|PAIA|HANA'
storms['maui_text_match'] = combined_text.str.contains(maui_terms, regex=True, na=False)

distance_cols = []
for lat_col, lon_col, out_col in [
    ('BEGIN_LAT','BEGIN_LON','begin_dist_ogg_km'),
    ('END_LAT','END_LON','end_dist_ogg_km'),
]:
    if lat_col in storms.columns and lon_col in storms.columns:
        storms[out_col] = [
            haversine_km(OGG_LAT, OGG_LON, la, lo)
            for la, lo in zip(storms[lat_col], storms[lon_col])
        ]
        distance_cols.append(out_col)

if distance_cols:
    storms['near_ogg_100km'] = storms[distance_cols].min(axis=1) <= 100
else:
    storms['near_ogg_100km'] = False

storms['maui_relevant'] = storms['maui_text_match'] | storms['near_ogg_100km']
maui_storms = storms[storms['maui_relevant']].copy()

print('All Hawaii events:', len(storms))
print('Maui/OGG-relevant events:', len(maui_storms))
if 'EVENT_TYPE' in maui_storms.columns:
    display(maui_storms['EVENT_TYPE'].value_counts().head(30).to_frame('count'))


## 7. Parse storm start/end times

NOAA Storm Events date strings are not guaranteed to use a single representation, so we use `format='mixed'` where supported.


In [ ]:
for src, dst in [('BEGIN_DATE_TIME','storm_begin_dt'), ('END_DATE_TIME','storm_end_dt')]:
    if src not in maui_storms.columns:
        raise KeyError(f'Missing required Storm Events column: {src}')
    try:
        maui_storms[dst] = pd.to_datetime(maui_storms[src], format='mixed', errors='coerce')
    except TypeError:
        maui_storms[dst] = pd.to_datetime(maui_storms[src], errors='coerce')

valid_storms = maui_storms.dropna(subset=['storm_begin_dt','storm_end_dt']).copy()
valid_storms = valid_storms[valid_storms['storm_end_dt'] >= valid_storms['storm_begin_dt']]

print('Valid timed Maui/OGG events:', len(valid_storms))
display(valid_storms[['EVENT_TYPE','storm_begin_dt','storm_end_dt']].head() if 'EVENT_TYPE' in valid_storms.columns else valid_storms[['storm_begin_dt','storm_end_dt']].head())


## 8. Attach active storm-event context to flights

A flight is storm-active when its scheduled OGG timestamp lies between the event's begin and end timestamps. Multiple simultaneous events are collapsed into a single row with count and joined event types.


In [ ]:
merged = merged.reset_index(drop=True)
merged['maui_storm_active'] = False
merged['maui_storm_count'] = 0
merged['maui_storm_types'] = ''

event_type_col = 'EVENT_TYPE' if 'EVENT_TYPE' in valid_storms.columns else None

for r in valid_storms.itertuples(index=False):
    begin = getattr(r, 'storm_begin_dt')
    end = getattr(r, 'storm_end_dt')
    mask = merged['ogg_sched_dt'].between(begin, end, inclusive='both')
    if not mask.any():
        continue
    merged.loc[mask, 'maui_storm_active'] = True
    merged.loc[mask, 'maui_storm_count'] += 1
    if event_type_col:
        et = str(getattr(r, event_type_col))
        current = merged.loc[mask, 'maui_storm_types']
        merged.loc[mask, 'maui_storm_types'] = np.where(
            current.eq(''), et, current + ' | ' + et
        )

print('Flights during Maui/OGG storm intervals:', int(merged['maui_storm_active'].sum()))
print('Percent:', round(100 * merged['maui_storm_active'].mean(), 3))


## 9. Add target and calendar features


In [ ]:
arr_delay_col = 'ArrDelay' if 'ArrDelay' in merged.columns else 'ArrDelayMinutes'

def disruption_class(r):
    if r.get('Cancelled', 0) == 1:
        return 'cancelled'
    d = r.get(arr_delay_col, np.nan)
    if pd.isna(d):
        return 'unknown'
    if d < 15:
        return 'normal'
    if d < 180:
        return 'delay'
    return 'severe_delay'

merged['disruption_class'] = merged.apply(disruption_class, axis=1)
merged['year'] = merged['ogg_sched_dt'].dt.year
merged['month'] = merged['ogg_sched_dt'].dt.month
merged['hour'] = merged['ogg_sched_dt'].dt.hour
merged['dayofweek'] = merged['ogg_sched_dt'].dt.dayofweek
merged['covid_era'] = merged['ogg_sched_dt'].between('2020-03-01', '2021-05-31')

display(merged['disruption_class'].value_counts().to_frame('count'))
display(pd.crosstab(merged['maui_storm_active'], merged['disruption_class'], normalize='index').mul(100).round(2))


## 10. Coverage diagnostics

Because the current NOAA LCD download ends in 2025 while BTS flights extend into 2026, 2026 flights are expected to lack matched weather. We quantify this explicitly rather than silently dropping them.


In [ ]:
coverage = merged.groupby('year').agg(
    flights=('ogg_sched_dt','size'),
    weather_matched=('weather_dt', lambda s: s.notna().sum()),
    storm_active=('maui_storm_active','sum'),
)
coverage['weather_match_pct'] = 100 * coverage['weather_matched'] / coverage['flights']
display(coverage.round(2))

print('Rows before optional weather restriction:', len(merged))
model_ready = merged[merged['weather_dt'].notna()].copy()
print('Rows with matched weather:', len(model_ready))


## 11. Save the merged dataset


In [ ]:
OUT_FILE.parent.mkdir(parents=True, exist_ok=True)
model_ready.to_csv(OUT_FILE, index=False, compression='gzip')
print('Saved:', OUT_FILE)
print('Shape:', model_ready.shape)


## Next notebook

Notebook 04 will use this merged table to:
- compare disrupted vs normal flights;
- test weather and airline predictors;
- create leakage-safe train/validation/test splits by time;
- train the first baseline classifier;
- evaluate calibration and explainability.
